# Лабораторная работа 5. Позиционное кодирование и RoPE (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 2:** attention и архитектура трансформера

**Цель.** Понять, как трансформер получает информацию о порядке токенов и почему модели плохо работают на длинах контекста, которых не видели при обучении.

**Что нужно сдать:**
1. Единый интерфейс для пяти вариантов: NoPE, синусоидального, обучаемого, ALiBi и RoPE.
2. Численную проверку относительности RoPE и график затухания верхней границы.
3. Таблицу «кодирование × длина контекста» для символьной LM (экстраполяция длины).
4. Position interpolation и NTK-aware scaling для RoPE без дообучения и после короткого дообучения.
5. Тест «иголка в стоге сена» на синтетической задаче поиска.

**Теория.** RoPE поворачивает пары координат \((x_{2i}, x_{2i+1})\) на угол \(m\theta_i\), где \(\theta_i = b^{-2i/d}\), а \(m\) — позиция. В комплексной записи \(\tilde q_m = q\,e^{im\theta}\), \(\tilde k_n = k\,e^{in\theta}\), поэтому
\[
\mathrm{Re}\langle \tilde q_m, \tilde k_n\rangle = \mathrm{Re}\sum_i q_i \bar k_i\, e^{i(m-n)\theta_i},
\]
то есть скалярное произведение зависит только от разности позиций \(m - n\).

**Расширение контекста в \(s\) раз:**
- position interpolation: \(m \mapsto m/s\);
- NTK-aware scaling: \(b \mapsto b\, s^{d/(d-2)}\) — высокие частоты почти не меняются, низкие интерполируются.

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 1. Позиционные кодирования

In [ ]:
def sinusoidal_pe(T, d):
    # TODO: ваш код
    raise NotImplementedError

def rope_angles(T, d, base=10000.0, scale=1.0):
    """Углы m*theta_i, форма (T, d/2). scale > 1 — position interpolation."""
    # TODO: ваш код
    raise NotImplementedError

def apply_rope(x, ang):
    """x: (..., T, d), ang: (T, d/2). Поворачивает пары (x[2i], x[2i+1])."""
    # TODO: ваш код
    raise NotImplementedError

def alibi_bias(T, n_head):
    """Смещение -m_h * (i - j) для j <= i; m_h = 2^{-8h/H}, h = 1..H. Форма (H, T, T)."""
    # TODO: ваш код
    raise NotImplementedError

### Проверка относительности RoPE

Для случайных \(q, k\) проверьте, что \(\langle R_m q, R_n k\rangle\) зависит только от \(m - n\), и постройте \(\frac{1}{d/2}\big|\sum_i e^{i r \theta_i}\big|\) в зависимости от расстояния \(r\) — величину, определяющую затухание верхней границы скалярного произведения при одинаковых \(q, k\).

In [ ]:
d = 64; torch.manual_seed(0); q, k = torch.randn(d), torch.randn(d)
ang = rope_angles(2048, d)
def rot_dot(m, n): return (apply_rope(q[None], ang[m:m + 1])[0] @ apply_rope(k[None], ang[n:n + 1])[0]).item()
for (m, n) in [(5, 2), (105, 102), (1005, 1002)]:
    print(f"m={m:4d}, n={n:4d}: <R_m q, R_n k> = {rot_dot(m, n):.6f}")
assert abs(rot_dot(5, 2) - rot_dot(1005, 1002)) < 1e-3

# TODO: ваш код (постройте |mean_i exp(i r theta_i)| для r = 0..2048 при base=10000 и base=500)

## 2. Трансформер с переключаемым позиционным кодированием

In [ ]:
class PEAttention(nn.Module):
    def __init__(self, d_model, n_head, pe):
        super().__init__()
        self.h, self.dh, self.pe = n_head, d_model // n_head, pe
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False); self.o = nn.Linear(d_model, d_model, bias=False)
        self.rope_base, self.rope_scale = 10000.0, 1.0
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.h, self.dh).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        # TODO: ваш код (для rope поверните q и k; для alibi передайте смещение + каузальную маску через attn_mask)
        raise NotImplementedError
        return self.o(y.transpose(1, 2).reshape(B, T, C))

class PETransformer(nn.Module):
    def __init__(self, vocab, pe, d_model=128, n_head=4, n_layer=2, max_T=64, mlp=True):
        super().__init__()
        self.pe, self.max_T = pe, max_T
        self.emb = nn.Embedding(vocab, d_model)
        if pe == "learned": self.pos = nn.Parameter(torch.randn(max_T, d_model) * 0.02)
        self.attn = nn.ModuleList([PEAttention(d_model, n_head, pe) for _ in range(n_layer)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layer)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layer)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
                                  if mlp else nn.Identity() for _ in range(n_layer)])
        self.use_mlp = mlp
        self.lnf, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, vocab)
    def forward(self, x):
        T = x.size(1); h = self.emb(x)
        if self.pe == "sinusoidal": h = h + sinusoidal_pe(T, h.size(-1)).to(x.device)
        if self.pe == "learned":
            if T > self.max_T: raise ValueError("обучаемые позиции не определены за пределами обучающей длины")
            h = h + self.pos[:T]
        for a, l1, l2, m in zip(self.attn, self.ln1, self.ln2, self.mlp):
            h = h + a(l1(h))
            if self.use_mlp: h = h + m(l2(h))
        return self.head(self.lnf(h))
    def set_rope(self, base=10000.0, scale=1.0):
        for a in self.attn: a.rope_base, a.rope_scale = base, scale

PES = ["nope", "sinusoidal", "learned", "alibi", "rope"]

## 3. Экстраполяция длины на символьной LM

Обучите модели на окнах длины `T_TRAIN` и измерьте среднюю кросс-энтропию на окнах длины \(T_{train}\), \(2T_{train}\), \(4T_{train}\), \(8T_{train}\). Отдельно посчитайте loss только на позициях \(\ge T_{train}\) — это и есть качество экстраполяции.

In [ ]:
text = load_shakespeare(); chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text]); ntr = int(0.9 * len(data)); train_d, val_d = data[:ntr], data[ntr:]
T_TRAIN = 64 if FAST_DEV_RUN else 128
STEPS = 400 if FAST_DEV_RUN else 4000

def lm_batch(d, T, B=32):
    ix = torch.randint(len(d) - T - 1, (B,))
    return torch.stack([d[i:i + T] for i in ix]).to(DEVICE), torch.stack([d[i + 1:i + T + 1] for i in ix]).to(DEVICE)

def train_lm(model, steps, T, lr=2e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    for s in range(steps):
        x, y = lm_batch(train_d, T); loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    return model

@torch.no_grad()
def eval_lengths(model, lengths, n_windows=16):
    # TODO: ваш код (для каждой длины L — средний loss по всем позициям и по позициям >= T_TRAIN)
    raise NotImplementedError

lengths = [T_TRAIN * m for m in (1, 2, 4, 8)]
lm_models, rows = {}, []
for pe in PES:
    set_seed(); t0 = time.time()
    lm_models[pe] = train_lm(PETransformer(len(chars), pe, max_T=T_TRAIN).to(DEVICE), STEPS, T_TRAIN)
    res = eval_lengths(lm_models[pe], lengths)
    rows.append({"PE": pe, **{f"L={L}": round(v[0], 3) for L, v in res.items()}, **{f"хвост L={L}": round(v[1], 3) for L, v in res.items() if L > T_TRAIN}})
    print(pe, f"{time.time() - t0:.0f} c")
import pandas as pd
pd.DataFrame(rows).set_index("PE")

## 4. Расширение контекста RoPE: position interpolation и NTK-aware scaling

Для RoPE-модели при длине \(L = s\,T_{train}\) сравните:
1. исходную модель;
2. position interpolation (`scale = s`);
3. NTK-aware scaling (`base = 10000 · s^{d/(d-2)}`).

Затем дообучите копии модели на длине \(L\) в течение короткого числа шагов (около 5% исходного обучения) с каждым вариантом и сравните снова.

In [ ]:
import copy
s_ext = 4; L_ext = s_ext * T_TRAIN; d_head = 128 // 4
variants = {"исходная": dict(base=10000.0, scale=1.0), "PI": dict(base=10000.0, scale=float(s_ext)),
            "NTK": dict(base=10000.0 * s_ext ** (d_head / (d_head - 2)), scale=1.0)}
# TODO: ваш код (оцените варианты без дообучения и после короткого дообучения на длине L_ext)

## 5. «Иголка в стоге сена»

Задача: последовательность пар `ключ значение` и в конце запрос-ключ; нужно вернуть значение (как в ЛР 4). Обучите attention-only модели (2 слоя, без MLP) на длине 32 и проверьте точность на длинах 32, 64, 128 (в полном режиме и 256). Запрос может относиться к паре в любом месте контекста.

In [ ]:
VOCAB_KV = 64

def needle_batch(T, B=128):
    # TODO: ваш код (n=(T-1)//2 пар; ключи различны в примере; цель — значение пары с ключом-запросом)
    raise NotImplementedError

T_NEEDLE = 33
test_T = [33, 65, 129] if FAST_DEV_RUN else [33, 65, 129, 257]
rows = []
for pe in PES:
    set_seed(); m = PETransformer(VOCAB_KV, pe, n_layer=2, max_T=T_NEEDLE, mlp=False).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for s in range(800 if FAST_DEV_RUN else 4000):
        x, y = needle_batch(T_NEEDLE); loss = F.cross_entropy(m(x)[:, -1], y)
        opt.zero_grad(); loss.backward(); opt.step()
    r = {"PE": pe}
    with torch.no_grad():
        for T in test_T:
            try:
                x, y = needle_batch(T, B=512); r[f"T={T}"] = (m(x)[:, -1].argmax(-1) == y).float().mean().item()
            except ValueError:
                r[f"T={T}"] = float("nan")
    rows.append(r); print(r)
pd.DataFrame(rows).set_index("PE").round(3)

## Вопросы

1. Докажите свойство относительности RoPE. Почему RoPE применяется к \(q\) и \(k\), но не к \(v\)?
2. Почему обучаемые позиционные эмбеддинги в принципе не экстраполируют, а синусоидальные формально определены, но на практике тоже плохо экстраполируют?
3. Как NoPE получает позиционную информацию в каузальной модели?
4. Объясните разницу между PI и NTK-aware scaling в терминах частот \(\theta_i\). Какие частоты «ломаются» при экстраполяции?
5. Как ALiBi реализует индуктивное смещение к локальности и почему оно мешает задаче поиска далёкой «иголки»?